# 练习二：缺失值处理 + merge 合并

> 任务三「AI 工具初体验」的 pandas 部分（2 / 2）
> 数据：`data/orders.csv`（40 条订单）、`data/customers.csv`（14 条客户）
> 这两份数据**故意做了手脚**：订单里有 5 个缺失值，客户表里有 2 个重复 ID。

## 这一节要回答一个问题

**「每个城市的订单总金额是多少？」**

听起来只要 merge 一下再 groupby。但这两份脏数据会让答案**在你不察觉的情况下变错**。
这一节的重点不是"怎么写 merge"，是**怎么发现 merge 写错了**。

In [1]:
from pathlib import Path

import pandas as pd

# 兼容两种打开方式：在项目根目录打开（路径是 pandas/data），
# 或者直接打开 pandas/ 目录（路径是 data）。两种都不用改代码。
DATA = Path("pandas/data")
if not DATA.exists():
    DATA = Path("data")

print("pandas", pd.__version__)
print("数据目录:", DATA.resolve())

pandas 3.0.6
数据目录: C:\Users\Administrator\WorkBuddy\2026-10-13-16-35-32\dev-basics\pandas\data


## 第 1 步：先看缺失值，别急着删

拿到一张表的第一件事，不是 `dropna()`，是**看清楚缺在哪、缺多少**。

In [2]:
orders = pd.read_csv(DATA / "orders.csv", encoding="utf-8-sig")
customers = pd.read_csv(DATA / "customers.csv", encoding="utf-8-sig")

print("orders:", orders.shape)
print("customers:", customers.shape)
orders.head()

orders: (40, 4)
customers: (14, 3)


,order_id,customer_id,amount,order_date
0,1000,5.0,320.0,2026-09-25
1,1001,4.0,596.0,2026-09-12
2,1002,4.0,636.0,2026-09-12
3,1003,NaN,73.0,2026-09-14
4,1004,2.0,458.0,2026-09-26


In [3]:
missing = orders.isna().sum()
pct = (orders.isna().mean() * 100).round(1)

summary = pd.DataFrame({"缺失数": missing, "占比%": pct})
summary[summary["缺失数"] > 0]

,缺失数,占比%
customer_id,3,7.5
amount,2,5.0


`customer_id` 缺 3 个（7.5%），`amount` 缺 2 个（5.0%）。

**关键判断**：这两个缺失的**性质完全不同**，处理方式也不该一样。

| 字段 | 缺意味着什么 | 该怎么处理 |
|---|---|---|
| `customer_id` | 这单不知道是谁下的 | **不能猜**，只能标记或排除 |
| `amount` | 金额没记上（系统丢了/回调失败） | 可以估，但**绝不能填 0** |

为什么不能填 0？看下一步。

## 第 2 步：`dropna` 的三种姿势

`dropna()` 不带参数是最暴力的：**只要这一行有任何空值，整行删掉**。

In [4]:
print("原始行数:                    ", len(orders))
print("dropna() 全列:                ", len(orders.dropna()))
print("dropna(subset=[customer_id]): ", len(orders.dropna(subset=["customer_id"])))
print("dropna(subset=[amount]):      ", len(orders.dropna(subset=["amount"])))

原始行数:                     40
dropna() 全列:                 35
dropna(subset=[customer_id]):  37
dropna(subset=[amount]):       38


40 行变 35 行——**一次删掉 12.5% 的数据**。而其中有些行只是金额缺失，客户信息是完整的。

**原则**：`dropna()` 一定要带 `subset`，明确"我按哪一列判断"。
不带参数等于说"我要完整数据"，那通常不是你的本意。

## 第 3 步：`fillna` 填什么，比填不填更重要

如果决定填，先看这一列的分布，再决定填什么值。

In [5]:
print(orders["amount"].describe().round(1).to_string())
print()
print("中位数:", orders["amount"].median())

count     38.0
mean     480.8
std      237.1
min       70.0
25%      309.2
50%      461.0
75%      663.0
max      884.0

中位数: 461.0


现在做一件事：分别用「填 0」「填中位数」「删掉缺失行」算**三个指标**——总金额、平均值、计数。

注意看：**哪一个指标完全不受影响？**

In [6]:
def summarize(s, label):
    return {
        "处理方式": label,
        "总金额": round(s.sum(), 1),
        "平均值": round(s.mean(), 2),
        "计数": s.count(),
    }


pd.DataFrame([
    summarize(orders["amount"].fillna(0), "填 0"),
    summarize(orders["amount"].fillna(orders["amount"].median()), "填中位数"),
    summarize(orders["amount"].dropna(), "删除缺失行"),
])

,处理方式,总金额,平均值,计数
0,填 0,18270.0,456.75,40
1,填中位数,19192.0,479.80,40
2,删除缺失行,18270.0,480.79,38


**看到了吗——「填 0」和「删除缺失行」的总金额完全相同，都是 18270.0。**

这不是巧合，是 `sum()` 的机制：它本来就会跳过 `NaN`。把 `NaN` 换成 `0`，加 0 不改变和。
所以**只核对总金额，你永远发现不了「填 0」有问题**。

真正出错的是另外两列：

| 指标 | 填 0 | 删除缺失行 | 差在哪里 |
|---|---|---|---|
| 总金额 | 18270.0 | 18270.0 | 一样（`sum` 跳过 `NaN`） |
| **平均值** | **456.75** | **480.79** | 填 0 把 2 笔"未知"当成 2 笔"0 元订单"，分母从 38 变 40，均值被拉低约 5% |
| **计数** | **40** | **38** | 你会以为有 40 笔金额记录，实际只有 38 笔 |

**这就是脏数据最阴的地方：它不报错，而且错在你没在看的那个指标上。**
很多报表只报总额，这个错误就会一直躺着，谁也不会发现。

**所以真正的问题不是"填 0 让总额变小"（不会），而是"填 0 改变了这一列的含义"**——
0 表示"这单没花钱"，`NaN` 表示"不知道花了多少"。这两件事被混成一件，后面所有统计都失去意义。

**为什么填中位数比填 0 好**：中位数代表"典型订单金额"，把 2 个缺失补成 461 元，
均值 479.8 非常接近删行后的 480.79，说明它基本没扭曲分布。

但严格说它也是编的——所以更严谨的做法是**加一列标记**，让看数据的人知道哪些值是填的：

In [7]:
orders["amount_is_missing"] = orders["amount"].isna()
orders["amount_filled"] = orders["amount"].fillna(orders["amount"].median()).round(1)

orders[["order_id", "amount", "amount_is_missing", "amount_filled"]].head(8)

,order_id,amount,amount_is_missing,amount_filled
0,1000,320.0,False,320.0
1,1001,596.0,False,596.0
2,1002,636.0,False,636.0
3,1003,73.0,False,73.0
4,1004,458.0,False,458.0
5,1005,NaN,True,461.0
6,1006,353.0,False,353.0
7,1007,640.0,False,640.0


**这就是"标记法"**：原始值保留，另开一列记"这个值是填的"，再另开一列放填过的值。

看起来啰嗦，但好处是**你可以随时切回真实数据**，也能在报告里注明"其中 2 单金额为估算"。

## 第 4 步：merge 之前，必须查主键唯一性 ⭐

这是这一节**最重要的一步**。很多人合并前不查，合并后发现数字不对，却完全不知道从哪查起。

查法只有一行：**`nunique()` 和行数比**。

In [8]:
print("customers 总行数:      ", len(customers))
print("customer_id 唯一值个数:", customers["customer_id"].nunique())
print("差额（重复的行数）:     ", len(customers) - customers["customer_id"].nunique())

customers 总行数:       14
customer_id 唯一值个数: 12
差额（重复的行数）:      2


In [9]:
dup = customers[customers["customer_id"].duplicated(keep=False)]
dup.sort_values("customer_id")

,customer_id,name,city
3,4,客户04,上海
12,4,客户04,上海
6,7,客户07,上海
13,7,客户07,北京


**客户 4 有两条完全一样的记录，客户 7 有两条城市不一样的记录。**

这不是我写数据写错了，是现实：同一家公司被录了两次，或者系统之间同步出了问题。
`duplicated(keep=False)` 的意思是"把**所有**重复的都标出来"，`keep="first"` 只会标后面的，
你会以为只有一条是坏的。

## 第 5 步：直接 merge，看行数怎么变

现在合并。**先看行数，再看内容。**

In [10]:
merged = orders.merge(customers, on="customer_id", how="inner")

print("订单原始行数:", len(orders))
print("merge 后行数:", len(merged))
print("多了几行:    ", len(merged) - len(orders))

订单原始行数: 40
merge 后行数: 44
多了几行:     4


**40 行的订单，merge 完变成 44 行。**

多出来的 4 行不是新数据，是**客户 4 和客户 7 的订单各被复制了一份**：

```
订单 #1023  客户4  ──┐
                      ├─→ 匹配到「客户4 上海」  → 1 行
                      └─→ 匹配到「客户4 上海」(重复) → 又 1 行     ← 订单变 2 行
```

这叫**一对多合并导致的行数膨胀**。后果是你后面 `sum()` 的时候，
客户 4 和 7 的订单金额会被**算两遍**。

**这就是为什么前面必须先查重复。** 如果没查，你只会看到"总金额好像偏大"，然后完全找不到原因。

## 第 6 步：去重后再 merge

到底保留哪一条？这是**业务决定**，不是技术决定。这里用 `keep="first"`（保留先出现的），
实际工作中你可能要按"更新时间最新"或"数据来源优先级"来选。

关键是：**去重这个动作必须是显式的、写出来的**，不能指望 merge 帮你做。

In [11]:
customers_clean = customers.drop_duplicates(subset=["customer_id"], keep="first")
print("去重后客户数:", len(customers_clean))

merged = orders.merge(customers_clean, on="customer_id", how="left")
print("merge 后行数:", len(merged))

# 行数必须和左表一致（因为左表是 orders，且右表主键已唯一）
assert len(merged) == len(orders), "行数对不上，说明还有一对多"
print("行数校验通过")

去重后客户数: 12
merge 后行数: 40
行数校验通过


**注意 `how="left"`**：用 inner 的话，`customer_id` 缺失的那 3 单会被直接丢掉，
你就永远看不到"有 3 单不知道是谁下的"这件事了。

**默认是 `how="inner"`，它会静默地删行。** 所以除非你确定不要未匹配的数据，否则一律写 `how="left"`。

## 第 7 步：查"谁没匹配上"

`indicator=True` 会多一列 `_merge`，标出每行的来源。

In [12]:
checked = orders.merge(customers_clean, on="customer_id", how="left", indicator=True)
checked["_merge"].value_counts()

_merge
both          37
left_only      3
right_only     0
Name: count, dtype: int64

37 条匹配上，3 条只有左边（就是 `customer_id` 缺失的那 3 单）。
`right_only` 是 0，说明客户表里没有"从没下过单"的人——这里没有，不代表现实中不会有。

## 第 8 步：现在才回答最初的问题

"每个城市的订单总金额是多少？"

In [13]:
by_city = (
    merged.groupby("city", dropna=False)
    .agg(订单数=("order_id", "count"), 总金额=("amount", "sum"))
    .sort_values("总金额", ascending=False)
)
by_city

,订单数,总金额
city,,
深圳,13,5239.0
上海,9,3954.0
杭州,6,3505.0
广州,6,2710.0
北京,3,1582.0
NaN,3,1280.0


注意三件事：

1. **`dropna=False`**：不写它，`city` 为空的那 3 单会被**直接跳过**，你就少算了一部分。
   写了它，会多出一行 `NaN`——这才是诚实的：确实有订单不知道属于哪个城市。
2. **`count` vs `sum`**：`count` 数的是"非空的个数"，`sum` 把空值当 0。所以两列的含义不同。
3. **总金额加起来等于多少？** 自己核一下，看和前面那张对比表的 **18270.0** 对不对得上。
   对不上就说明有订单被漏掉了——大概率是 `city` 为空的那 3 单。

## 小结：这一节的三条铁律

| 铁律 | 为什么 |
|---|---|
| **`dropna()` 必须带 `subset`** | 不带等于"删任何有空的行"，通常会删掉你不该删的 |
| **金额类缺失不能填 0** | 0 是"没有"，不是"不知道"。总额可能不变（`sum` 跳过 `NaN`），但**均值和计数全错** |
| **merge 前必查主键唯一性** | 不查就会行数膨胀、金额算两遍，而且事后极难定位 |

**一个可以背下来的检查清单**：

```
merge 之前：右表主键 unique 吗？        → len(df) == df[key].nunique()
merge 之后：行数对不对得上？            → len(merged) == len(左表)
           未匹配的有多少？             → indicator=True，看 _merge
```

## 自己动手（2 个小任务）

1. 把 `orders` 里 `customer_id` 缺失的 3 单**单独存成一张表**，并数一下它们的总金额。
   （提示：`orders[orders["customer_id"].isna()]`）
2. 换一种去重策略：客户 7 有两条城市不同的记录，如果按"保留最后一条"（`keep="last"`），
   最终每个城市的金额会怎么变？**先猜，再跑，看猜得对不对。**